In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.functions import udf

@udf(returnType=IntegerType())
def convert_time_to_ms(time_str):
    if time_str is None:
        return None
    try:
        if ":" in time_str:
            parts = time_str.split(":")
            minutes = int(parts[0])
            sec_parts = parts[1].split(".")
            seconds = int(sec_parts[0])
            milliseconds = int(sec_parts[1]) if len(sec_parts) > 1 else 0
            return (minutes * 60 * 1000) + (seconds * 1000) + milliseconds
        else:
            sec_parts = time_str.split(".")
            seconds = int(sec_parts[0])
            milliseconds = int(sec_parts[1]) if len(sec_parts) > 1 else 0
            return (seconds * 1000) + milliseconds
    except (ValueError, IndexError):
        return None

def races_specific(df):
    df = df.drop("year")
    time_cols = ["time", "fp1_time", "fp2_time", "fp3_time", "quali_time", "sprint_time"]
    for col in time_cols:
        if col in df.columns:
            df = df.withColumn(col, F.date_format(F.try_to_timestamp(F.col(col), "HH:mm:ss"), "HH:mm:ss"))
    return df

def results_specific(df):
    df = df.withColumn("fastestlapspeed", F.col("fastestlapspeed").try_cast(DoubleType()))
    df = df.withColumn("fastestlap", F.col("fastestlap").try_cast(IntegerType()))
    df = df.withColumn("fastestlaptime", convert_time_to_ms(F.col("fastestlaptime")))
    df = df.drop("time")
    return df

def pit_stops_specific(df):
    df = df.withColumn("duration", convert_time_to_ms(F.col("duration")))
    df = df.withColumn("time", F.date_format(F.try_to_timestamp(F.col("time"), "HH:mm:ss"), "HH:mm:ss"))
    return df

def lap_times_specific(df):
    df = df.drop("time")
    return df

def qualifying_specific(df):
    for col in ["q1", "q2", "q3"]:
        if col in df.columns:
            df = df.withColumn(col, convert_time_to_ms(F.col(col)))
    return df

def sprint_results_specific(df):
    df = df.withColumn("fastestlaptime", convert_time_to_ms(F.col("fastestlaptime")))
    df = df.withColumn("fastestlap", F.col("fastestlap").try_cast(IntegerType()))
    df = df.withColumn("rank", F.col("rank").try_cast(IntegerType()))
    df = df.drop("time")
    return df

def seasons_specific(df):
    df = df.withColumn("year", F.to_date(F.col("year").cast("string"), "yyyy"))
    return df

def safety_car_specific(df):
    df = df.withColumn("cause",
        F.when(F.col("cause") == "multiple accidents", F.col("cause"))
         .when(F.col("cause").rlike("(?i)^accident.*"), "accident")
         .when(F.col("cause").rlike("(?i)^stranded car.*"), "stranded car")
         .otherwise(F.col("cause"))
    )
    return df

def apply_specific_silver_transforms(df, table_name):
    f1_transform_map = {
        "race_data_races": races_specific,
        "race_data_results": results_specific,
        "race_data_pit_stops": pit_stops_specific,
        "race_data_lap_times": lap_times_specific,
        "race_data_qualifying": qualifying_specific,
        "race_data_sprint_results": sprint_results_specific,
        "race_data_seasons": seasons_specific,
        "race_events_safety_car": safety_car_specific
    }
    if table_name in f1_transform_map:
        df = f1_transform_map[table_name](df)
    return df